# Colab Session — Cedar

Installs OMPK and joins your Tailscale network. No model server or GPU inference is provisioned.

In Colab's **Secrets** panel (key icon), add `TAILSCALE_AUTH_KEY` and `OPENROUTER_API_KEY`, and enable notebook access for both. Use a short-lived, preferably ephemeral Tailscale auth key with permission to join your tailnet. Never paste keys into notebook cells.

Run the cells in order. Secrets remain in runtime memory, not notebook output or saved configuration. Colab runtimes are temporary; rerun setup after a reset. Tailscale uses userspace networking: this joins the tailnet but does not automatically expose SSH or route every process through Tailscale.


In [ ]:
import os
import subprocess
import shutil
import time
import tempfile
import json
from pathlib import Path
from google.colab import userdata

# Read both secrets before changing the runtime. Never print their values.
for name in ("TAILSCALE_AUTH_KEY", "OPENROUTER_API_KEY"):
    value = userdata.get(name)
    if not value or not value.strip():
        raise RuntimeError(f"Set {name} in Colab Secrets and enable notebook access.")
    os.environ[name] = value.strip()
del value

# Installers and OMPK do not need the Tailscale key in their environment.
def run_checked(args, *, timeout=300):
    child_env = os.environ.copy()
    child_env.pop("TAILSCALE_AUTH_KEY", None)
    result = subprocess.run(args, env=child_env, capture_output=True, text=True, timeout=timeout)
    if result.returncode:
        # Do not dump stderr: authentication commands may echo credentials.
        raise RuntimeError(f"{Path(args[0]).name} failed (exit {result.returncode}); credential-bearing output withheld.")
    return result.stdout.strip()

print("Colab secrets loaded.")


In [ ]:
# Install the published OMPK binary using the project's official installer.
# Download first so a failed download cannot look like a successful install.
run_checked(["curl", "-fsSL", "https://oh-my-pk.pkking.computer/install.sh", "-o", "/tmp/ompk-install.sh"])
run_checked(["sh", "/tmp/ompk-install.sh", "--binary"], timeout=600)
os.environ["PATH"] = ":".join([str(Path.home() / ".local/bin"), str(Path.home() / ".bun/bin"), os.environ["PATH"]])
ompk = shutil.which("ompk")
if not ompk:
    raise RuntimeError("Installer completed but ompk was not found on PATH.")
print("OMPK installed:", run_checked([ompk, "--version"]))

run_checked(["curl", "-fsSL", "https://tailscale.com/install.sh", "-o", "/tmp/tailscale-install.sh"])
run_checked(["sh", "/tmp/tailscale-install.sh"], timeout=600)
print("Tailscale installed:", run_checked(["tailscale", "version"]).splitlines()[0])


In [ ]:
# A dedicated userspace daemon avoids depending on Colab's systemd or /dev/net/tun.
state_dir = Path("/tmp/colab-session-cedar")
state_dir.mkdir(mode=0o700, exist_ok=True)
state_dir.chmod(0o700)
socket = str(state_dir / "tailscaled.sock")
ts = ["tailscale", f"--socket={socket}"]
probe = subprocess.run(ts + ["status", "--json"], capture_output=True, text=True)
if probe.returncode != 0:
    daemon_env = os.environ.copy()
    daemon_env.pop("TAILSCALE_AUTH_KEY", None)
    daemon_env.pop("OPENROUTER_API_KEY", None)
    with open(state_dir / "tailscaled.log", "ab") as log:
        tailscaled_process = subprocess.Popen([
            "tailscaled", "--tun=userspace-networking",
            f"--socket={socket}", f"--state={state_dir / 'tailscaled.state'}",
        ], stdout=log, stderr=log, env=daemon_env, start_new_session=True)
    for _ in range(30):
        if tailscaled_process.poll() is not None:
            raise RuntimeError("Tailscale daemon exited; inspect its local log (do not publish credentials).")
        probe = subprocess.run(ts + ["status", "--json"], capture_output=True, text=True)
        if probe.returncode == 0:
            break
        time.sleep(1)
    else:
        raise RuntimeError("Tailscale daemon did not become ready within 30 seconds.")

status = json.loads(run_checked(ts + ["status", "--json"]))
if status.get("BackendState") != "Running":
    # File-based auth keeps the key out of process arguments. Remove it even on failure.
    fd, key_path = tempfile.mkstemp(prefix="ts-auth-", dir=state_dir)
    try:
        with os.fdopen(fd, "w") as key_file:
            key_file.write(os.environ["TAILSCALE_AUTH_KEY"])
        run_checked(ts + ["up", f"--auth-key=file:{key_path}",
                          "--hostname=colab-session-cedar", "--accept-dns=false"], timeout=90)
    finally:
        Path(key_path).unlink(missing_ok=True)
status = json.loads(run_checked(ts + ["status", "--json"]))
if status.get("BackendState") != "Running":
    raise RuntimeError("Tailscale has not reached Running state.")
os.environ.pop("TAILSCALE_AUTH_KEY", None)
print("Tailscale connected:", ", ".join(status.get("TailscaleIPs", [])))
print("OpenRouter configured for OMPK through OPENROUTER_API_KEY.")


## Use OMPK

The next cell sends a real request to OpenRouter and may incur API charges. Replace the model ID with one available to your account. OMPK inherits `OPENROUTER_API_KEY` from the notebook runtime; no `/login` or plaintext credential file is needed.

For an interactive terminal in this same runtime, launch `ompk --provider openrouter --model <model-id>` from a terminal that inherits this environment. A separate SSH session does **not** automatically inherit notebook secrets. This notebook does not configure SSH, an OMPK bridge, or a persistent remote session.


In [ ]:
# Optional: run this cell to verify an actual OpenRouter generation.
MODEL = "openai/gpt-4o-mini"
reply = run_checked([ompk, "--provider", "openrouter", "--model", MODEL,
                     "--print", "Reply with exactly: Cedar ready"], timeout=180)
print(reply)
